# Optimizing a stellarator geometry from GVEC

This notebook demonstrates the GVEC → local flux-tube GK → optimizer interface. It is runnable without pyGVEC by selecting a provider-neutral synthetic field line; set the live switch to evaluate an actual GVEC state. Placeholder pieces are identified explicitly.

> Current capability: the GVEC adapter is an evaluation boundary and is marked non-differentiable. Geometry derivatives should therefore be taken in an outer finite-difference or derivative-free loop.

## Division of responsibility

`GVEC parameter/state` → equilibrium and PEST evaluation → `GvecGeometryProvider` → physical arrays (`B`, gradients, curvature, $\iota$, shear) → `GeometryResult` → normalized internal coefficients → linear gyrokinetic residual/time evolution → growth, frequency, mode structure, quasilinear proxy → scalar objective.

GVEC supplies a global equilibrium; `jax_fluxtube_gk` samples one local field line and predicts reduced kinetic stability there. A realistic stellarator objective usually aggregates several radii, field-line labels $\alpha$, and $k_y$ values, while adding equilibrium and engineering constraints outside the GK solver.

In [ ]:
import os
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from jax_fluxtube_gk import (
    DesignObjectiveSpec, FourierGridSpec, GeometryRequest, GvecGeometryProvider,
    OptimizationKnobs, SingleSurfaceOptimizationConfig, SyntheticGeometryProvider,
    VelocityGridSpec, build_fourier_grid, build_mode_connectivity, build_velocity_grid,
    design_objective, internal_geometry_from_result, k_perp_squared, resolve_geometry,
)

USE_LIVE_GVEC = False
GVEC_PARAMETER_FILE = os.environ.get("GVEC_PARAMETER_FILE")
GVEC_STATE_FILE = os.environ.get("GVEC_STATE_FILE")

In [ ]:
request = GeometryRequest(
    configuration="gvec-design", radial_value=0.8, alpha=0.0, n_z=32,
    z_min=-np.pi, z_max=np.pi, field_periods=1.0,
)
if USE_LIVE_GVEC:
    if not GVEC_PARAMETER_FILE:
        raise RuntimeError("Set GVEC_PARAMETER_FILE (and optionally GVEC_STATE_FILE)")
    provider = GvecGeometryProvider(
        parameter_file=GVEC_PARAMETER_FILE, state_file=GVEC_STATE_FILE, revision="record-your-revision"
    )
else:
    provider = SyntheticGeometryProvider(magnetic_field_amplitude=0.16, iota=0.85, shear=0.25, nfp=5)

geometry_result = resolve_geometry(provider, request)
parallel = geometry_result.parallel_grid
geometry = internal_geometry_from_result(geometry_result)
metadata = geometry_result.metadata
print(f"provider={metadata.provenance.provider!r}, differentiable={metadata.differentiable}")
print(f"source={metadata.provenance.source}")

In [ ]:
fourier = build_fourier_grid(FourierGridSpec(n_kx=1, n_ky=1, kx_max=0.0, ky_values=(0.3,)))
kp2 = np.asarray(k_perp_squared(geometry, fourier)).squeeze()
z = np.asarray(parallel.z)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), constrained_layout=True)
axes[0].plot(z, np.asarray(geometry.B)); axes[0].set_title(r"$B(z)$")
axes[1].plot(z, np.asarray(geometry.g_xx), label=r"$g_{xx}$")
axes[1].plot(z, np.asarray(geometry.g_yy), label=r"$g_{yy}$"); axes[1].legend()
axes[1].set_title("perpendicular metric")
axes[2].plot(z, kp2); axes[2].set_title(r"$k_\perp^2(z)$ at $k_y=0.3$")
for ax in axes: ax.set_xlabel("toroidal field-line coordinate zeta"); ax.grid(alpha=0.25)
plt.show()

## Trace one reduced GK objective

The state has axes `(v_parallel, mu, z, kx, ky)`. The linear residual combines parallel streaming, magnetic drifts, profile-gradient drive, field response, and the configured collision/closure pieces. After short time evolution, diagnostics turn the mode amplitude and phase into growth and frequency. Here the resolution is intentionally tiny so the notebook tests integration, not convergence.

In [ ]:
velocity = build_velocity_grid(VelocityGridSpec(n_vpar=2, n_mu=2, vpar_max=1.5, mu_max=1.0))
connectivity = build_mode_connectivity(fourier)
shape = (velocity.vpar.size, velocity.mu.size, parallel.z.size, 1, 1)
index = jnp.arange(np.prod(shape), dtype=jnp.float64).reshape(shape)
initial_state = 1e-2 * (jnp.cos(index / 7.0) + 1j * jnp.sin(index / 11.0))
spec = DesignObjectiveSpec(selected_ky=0, growth_weight=1.0)
config = SingleSurfaceOptimizationConfig(
    geometry_model="precomputed", dt=0.005, n_steps=2, selected_ky=0,
    objective_kind="selected_growth", store_history=False,
)

def gk_objective(local_geometry, temperature_gradient=2.1):
    return design_objective(
        OptimizationKnobs(density_gradient=0.8, temperature_gradient=temperature_gradient),
        velocity, parallel, fourier, initial_state, spec, connectivity=connectivity,
        config=config, geometry=local_geometry,
    )

result = gk_objective(geometry)
profile_grad = jax.grad(lambda a: gk_objective(geometry, a).scalar_objective)(jnp.array(2.1))
print(f"objective={float(result.scalar_objective): .6e}")
print(f"growth={float(result.selected_growth_rate): .6e}, frequency={float(result.selected_frequency): .6e}")
print(f"fixed-geometry d objective / d(R/L_T)={float(profile_grad): .6e}")

## Geometry loop for GVEC

For a real design vector $p$, construct a fresh GVEC state for $p$, evaluate the same fixed set of `(rho, alpha, ky)` samples, and aggregate their GK objectives. Central differences require `2N+1` equilibrium/GK evaluations for $N$ parameters; derivative-free methods can be preferable when equilibrium convergence is noisy. Always record convergence and constraint residuals, and penalize/reject failed equilibria.

The executable loop below is an **interface surrogate**. Its parameter changes synthetic magnetic shear and its cost is a cheap geometric proxy. Replace `surrogate_cost` by fresh `GvecGeometryProvider(...)` and `gk_objective(...)` calls for a real loop.

In [ ]:
def surrogate_cost(shear):
    candidate = resolve_geometry(
        SyntheticGeometryProvider(magnetic_field_amplitude=0.16, iota=0.85, shear=shear, nfp=5), request
    )
    geom = internal_geometry_from_result(candidate)
    return float(jnp.mean(k_perp_squared(geom, fourier)))

parameter, h, rate = 0.35, 1e-3, 1.0
history = []
for iteration in range(7):
    value = surrogate_cost(parameter)
    gradient = (surrogate_cost(parameter + h) - surrogate_cost(parameter - h)) / (2 * h)
    history.append((iteration, parameter, value, gradient))
    parameter = float(np.clip(parameter - rate * gradient, -1.0, 1.0))
history = np.asarray(history)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), constrained_layout=True)
axes[0].plot(history[:, 0], history[:, 1], "o-"); axes[0].set_ylabel("surrogate shear")
axes[1].plot(history[:, 0], history[:, 2], "o-"); axes[1].set_ylabel("proxy objective")
for ax in axes: ax.set_xlabel("iteration"); ax.grid(alpha=0.25)
plt.show()

In [ ]:
def evaluate_live_gvec_design(design_parameters):
    """Project-specific outer-loop seam; intentionally incomplete."""
    # 1. Map design_parameters to a GVEC parameter file/state and solve it.
    # 2. provider = GvecGeometryProvider(state=state, revision=...)
    # 3. result = resolve_geometry(provider, request)
    # 4. objective = gk_objective(internal_geometry_from_result(result))
    # 5. Return objective, MHD/engineering constraints, convergence, and provenance.
    raise NotImplementedError("Supply the project-specific GVEC design parameterization")

### Production checklist

Use a small symmetry-preserving parameter set first; freeze numerical topology; aggregate several flux tubes rather than trusting one; store provider version/revision and normalization; keep generated states in external scratch; audit finite-difference step size; and re-run accepted candidates with converged equilibrium and GK resolution.